In [0]:
import json
import requests
import re
from datetime import datetime
from pyspark.sql import SparkSession
from concurrent.futures import ThreadPoolExecutor, as_completed

# API Key
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

# ==========================================
# 1. AI Validation Engine (Stable Version)
# ==========================================
def validate_job_with_ai(job_title, company_name, job_description, apply_link):
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an Elite US IT Bench Sales Recruiter and Data Validator.
    Analyze this scraped web page to determine if it is a GENUINE job posting and extract its core details.

    Title: {job_title}
    Company: {company_name}
    Apply Link: {apply_link}
    Scraped Text: 
    {job_description}

    STRICT VALIDATION RULES:
    1. INVALIDATE NON-JOBS: If the link or text is from Wikipedia, W3Schools, GeeksforGeeks, Python documentation, blogs, or any tutorial site, mark "is_valid": false.
    2. INVALIDATE ERRORS: If the text says "Checking your browser", "Access Denied", "CAPTCHA", "Enable JavaScript", or is too short to be a real job, mark "is_valid": false.
    3. JOB TYPE: Analyze the text to find the exact job type. Choose ONLY from: ["W2", "C2C", "Contract", "Full-Time", "Not Found"]. If you cannot clearly identify the type, output "Not Found".
    4. CLIENT REQUIREMENTS: If valid, read the description and summarize EXACTLY what the client wants in 2-3 short, crisp bullet points (e.g., "1. 5+ yrs Python  2. PySpark & Delta Lake  3. AWS"). If invalid, leave as null.
    5. HR EMAIL: Extract any recruiter/HR email if present, else null.

    Return ONLY a valid JSON object matching exactly this structure:
    {{
        "is_valid": true,
        "job_type": "Full-Time",
        "client_requirements": "1. Minimum 5 years Python 2. AWS and CI/CD pipelines",
        "hr_email": "hr@company.com",
        "reasoning": "Clear Data Engineer requirement. Not a tutorial or captcha page."
    }}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1, # Extremely low temperature for logical strictness
        "max_tokens": 500
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            # Bulletproof Regex JSON Extractor
            match = re.search(r'\{.*\}', content, re.DOTALL)
            if match:
                try:
                    return json.loads(match.group(0))
                except json.JSONDecodeError:
                    cleaned = match.group(0).replace("'", '"').replace(",}", "}")
                    return json.loads(cleaned)
        return {"is_valid": False, "job_type": "Not Found", "hr_email": None, "reasoning": "AI Parsing Failed"}
    except Exception as e:
        print(f"❌ AI Call Failed: {str(e)}")
        return {"is_valid": False, "job_type": "Not Found", "hr_email": None, "reasoning": "Exception occurred"}

# ==========================================
# 2. Parallel Processing Workflow
# ==========================================
def process_single_validation(row_dict):
    job_id = row_dict['id']
    company = row_dict['company_name']
    title = row_dict['job_title']
    desc = row_dict['job_description']
    apply_link = row_dict['apply_link']
    
    # Send to AI
    ai_result = validate_job_with_ai(title, company, desc, apply_link)
    
    return {
        "job_id": job_id,
        "company": company,
        "title": title,
        "apply_link": apply_link,
        "ai_result": ai_result
    }

def run_stable_validation_pipeline():
    print("🚀 Starting Ultra-Stable AI Validation Pipeline...")
    
    # Let's take up to 20 Pending jobs for validation
    pending_jobs_df = spark.sql("""
        SELECT * FROM jobs_automation_db.default.raw_jobs_staging 
        WHERE validation_status = 'Pending'
    """)
    
    pending_count = pending_jobs_df.count()
    if pending_count == 0:
        print("✅ No pending jobs to validate right now.")
        return
        
    print(f"🔎 Found {pending_count} pending raw records. AI is filtering out the garbage...\n")
    
    pending_jobs = [row.asDict() for row in pending_jobs_df.collect()]
    validated_records = []
    processed_job_ids = []
    
    with ThreadPoolExecutor(max_workers=5) as executor:
        future_to_job = {executor.submit(process_single_validation, job): job for job in pending_jobs}
        
        for future in as_completed(future_to_job):
            result = future.result()
            job_id = result['job_id']
            ai_result = result['ai_result']
            company = result['company']
            title = result['title']
            
            if ai_result and ai_result.get("is_valid"):
                job_type = ai_result.get("job_type", "Not Found")
                reqs = ai_result.get("client_requirements", "Not extracted")
                
                print(f"   ✔️ VALID JOB! {title} @ {company} | Type: {job_type}")
                print(f"      📌 Client Wants: {reqs}")
                
                validated_records.append({
                    "job_id": job_id,
                    "company_name": company,
                    "job_title": title,
                    "job_type": job_type,
                    "is_valid": True,
                    "hr_email": ai_result.get("hr_email"),
                    "client_requirements": reqs,
                    "apply_url": result['apply_link'],
                    "validated_date": datetime.now()
                })
            else:
                reason = ai_result.get("reasoning", "No reason provided") if ai_result else "AI Failed"
                print(f"   ❌ REJECTED (Garbage/Invalid): {title} @ {company} | Reason: {reason}")
                
            processed_job_ids.append(f"'{job_id}'")
            
    # Save perfectly valid jobs to Master Table
    if len(validated_records) > 0:
        valid_df = spark.createDataFrame(validated_records)
        ordered_columns = [
            "job_id", "company_name", "job_title", "job_type", 
            "is_valid", "hr_email", "client_requirements", "apply_url", "validated_date"
        ]
        valid_df = valid_df.select(*ordered_columns)
        valid_df.write.mode("append").insertInto("jobs_automation_db.default.validated_jobs_master")
        print(f"\n💾 Saved {len(validated_records)} GOLDEN, FULLY VERIFIED jobs to master table.")
    else:
        print("\n🗑️ All analyzed jobs were garbage/spam. Nothing saved to master table.")
    
    # Update staging status so we don't process these again
    if processed_job_ids:
        ids_string = ",".join(processed_job_ids)
        spark.sql(f"""
            UPDATE jobs_automation_db.default.raw_jobs_staging 
            SET validation_status = 'Processed', updated_at = current_timestamp()
            WHERE id IN ({ids_string})
        """)
        print("🔄 Updated raw_jobs_staging status to 'Processed'.")

run_stable_validation_pipeline()


In [0]:
%sql
select * from jobs_automation_db.default.validated_jobs_master